# Research notebook archive

Original project notebook with outputs and personal metadata removed. Existing Colab paths and experiment settings require configuration; see the repository README before running.

# Stage 3 Sequential Fine-Tuning Experiment Pipeline

This notebook runs the Stage 3 sequential fine-tuning experiments across seven demographic attributes and three data construction methods.

It supports:

- `natural`, `balanced`, and `weighted` training methods
- five total experiment runs, not all method/order combinations
- a low-compute smoke test using a small model
- independent/resumable runs in case Colab disconnects
- LoRA checkpoint saving after every trained target
- validation evaluation after every training step on all targets
- final test evaluation after the full sequence

## Experimental design

Main method comparison:

| Run | Method | Order |
|---|---|---|
| `natural_canonical` | natural | canonical |
| `balanced_canonical` | balanced | canonical |
| `weighted_canonical` | weighted | canonical |

Recency/order analysis:

| Run | Method | Order |
|---|---|---|
| `natural_canonical` | natural | canonical |
| `natural_reverse` | natural | reverse |
| `natural_random1` | natural | random |

The natural canonical run is reused, so only **5 total sequential trainings** are needed.

## Why this is valid

The design separates two questions:

1. Do training data methods change demographic fidelity?
2. Does sequential order create recency bias or forgetting?

The method comparison holds order fixed. The recency comparison holds method fixed. This avoids confounding method effects with order effects while saving compute.

## How to use this notebook

1. Start with `SMOKE_TEST = True`.
2. Run the notebook once with the small model.
3. If it works, set `SMOKE_TEST = False`.
4. Set `RUN_NAME` to one run.
5. Run one experiment at a time.
6. If Colab disconnects, resume from the last saved adapter checkpoint.

## 1. Install packages

In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes

## 2. Imports

In [ ]:
import os
import json
import random
import string
import gc
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel,
    prepare_model_for_kbit_training,

)

from transformers import DataCollatorForLanguageModeling


In [ ]:
!pip uninstall -y torchao

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 4. Global configuration

Start in smoke-test mode. The smoke test uses a tiny model, tiny datasets, and only a few optimization steps so you can verify the full pipeline without using many compute units.

In [ ]:
# ============================================================
# MODE
# ============================================================

SMOKE_TEST = True

# ============================================================
# PATHS
# ============================================================

DATASET_ROOT = Path(
    "/content/drive/MyDrive/NLP_project/data/stage3_outputs/stage3_training_datasets"
)

EXPERIMENT_ROOT = Path(
    "/content/drive/MyDrive/NLP_project/data/stage3_outputs/sequential_experiments"
)

EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)

# ============================================================
# TARGETS
# ============================================================

TARGETS = [
    "SEX",
    "AGE_BINS",
    "RACE_ETHNICITY",
    "EDUCATION_LEVEL",
    "INCTOT_BINS",
    "HEALTH_INSURANCE",
    "ENGLISH_STATUS",
]

CANONICAL_ORDER = [
    "SEX",
    "AGE_BINS",
    "EDUCATION_LEVEL",
    "RACE_ETHNICITY",
    "INCTOT_BINS",
    "HEALTH_INSURANCE",
    "ENGLISH_STATUS",
]

REVERSE_ORDER = list(reversed(CANONICAL_ORDER))

RANDOM_ORDER_1 = [
    "RACE_ETHNICITY",
    "ENGLISH_STATUS",
    "AGE_BINS",
    "HEALTH_INSURANCE",
    "EDUCATION_LEVEL",
    "SEX",
    "INCTOT_BINS",
]

RUN_CONFIGS = {
    "natural_canonical": {
        "method": "natural",
        "order": CANONICAL_ORDER,
        "description": "Main natural run; also recency baseline.",
    },
    "balanced_canonical": {
        "method": "balanced",
        "order": CANONICAL_ORDER,
        "description": "Main balanced-sampling run.",
    },
    "weighted_canonical": {
        "method": "weighted",
        "order": CANONICAL_ORDER,
        "description": "Main weighted-loss run.",
    },
    "natural_reverse": {
        "method": "natural",
        "order": REVERSE_ORDER,
        "description": "Reverse-order recency check.",
    },
    "natural_random1": {
        "method": "natural",
        "order": RANDOM_ORDER_1,
        "description": "Random-order recency robustness check.",
    },
}

# Choose one run at a time.
RUN_NAME = "natural_canonical"

# Use balanced evaluation by default for subgroup-comparable checks.
# Change to "raw" for population-like validation/test.
VALIDATION_KIND = "balanced"
TEST_KIND = "balanced"

# ============================================================
# MODEL SETTINGS
# ============================================================

if SMOKE_TEST:
    MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

    # Tiny datasets for debugging only.
    DEBUG_MAX_TRAIN_ROWS_PER_TARGET = 24
    DEBUG_MAX_EVAL_ROWS_PER_TARGET = 24

    NUM_EPOCHS = 1
    MAX_STEPS_PER_TARGET = 2

    PER_DEVICE_BATCH_SIZE = 1
    GRAD_ACCUM = 1
    LEARNING_RATE = 2e-4

    USE_4BIT = False
else:
    MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"
    DEBUG_MAX_TRAIN_ROWS_PER_TARGET = None
    DEBUG_MAX_EVAL_ROWS_PER_TARGET = None

    NUM_EPOCHS = 1
    MAX_STEPS_PER_TARGET = -1

    PER_DEVICE_BATCH_SIZE = 16
    GRAD_ACCUM = 2
    LEARNING_RATE = 2e-4

    USE_4BIT = True

MAX_VALIDATION_ROWS_PER_TARGET = DEBUG_MAX_EVAL_ROWS_PER_TARGET
MAX_TEST_ROWS_PER_TARGET = DEBUG_MAX_EVAL_ROWS_PER_TARGET

SAVE_INTERMEDIATE_CHECKPOINTS = True
SAVE_FINAL_MERGED_MODEL = True

SEED = 42

## 5. Prompt construction

In [ ]:
FEATURE_LABELS = {
    "HHINCOME": "Household income",
    "PERWT": "Person statistical weight",
    "NCHILD": "Number of children",
    "SEX": "Sex",
    "AGE": "Age",
    "AGE_BINS": "Age group",
    "UHRSWORK": "Usual hours worked per week",
    "INCTOT": "Total personal income",
    "INCTOT_BINS": "Total personal income group",
    "INCWAGE": "Wage income",
    "STATE_NAME": "State",
    "OWNERSHP_STATUS": "Home ownership",
    "MARITAL_STATUS": "Marital status",
    "RACE_ETHNICITY": "Race/ethnicity",
    "ENGLISH_STATUS": "English speaking ability",
    "HEALTH_INSURANCE": "Health insurance coverage",
    "EMPLOYMENT_STATUS": "Employment status",
    "EDUCATION_LEVEL": "Education",
    "WORKER_CLASS": "Class of worker",
    "TRANSPORT_METHOD": "Transportation to work",
}

LABEL_TO_LONG = {
    "White": "White",
    "Hispanic": "Hispanic",
    "Black": "Black or African American",
    "Indigenous": "American Indian or Alaska Native",
    "Chinese": "Chinese",
    "Japanese": "Japanese",
    "Other": "Other Asian or Pacific Islander",
}

TARGET_OPTION_ORDER = {
    "RACE_ETHNICITY": [
        "White",
        "Hispanic",
        "Black",
        "Other",
        "Chinese",
        "Indigenous",
        "Japanese",
    ],
    "AGE_BINS": [
        "25–34",
        "35–44",
        "45–54",
        "55–64",
    ],
}

SYSTEM_PROMPT = """You are completing one missing demographic attribute for a U.S. survey respondent.

You will be given a profile with one missing attribute and a list of lettered answer options. Predict the most likely value of that missing attribute using the other provided attributes.

Rules:
- Use only the provided profile.
- Choose exactly one option from the list.
- Reply with only the letter of the selected option (A, B, C, ...) — no parentheses, no explanation, no extra text."""


def set_seed(seed=42):
    """Make training/evaluation behavior as reproducible as possible."""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def display_label(value):
    """Convert compact class labels into display labels when needed."""

    return LABEL_TO_LONG.get(value, value)


def get_target_classes(df, target_col):
    """Return class labels in a stable order."""

    observed = df[target_col].dropna().unique().tolist()

    if target_col in TARGET_OPTION_ORDER:
        ordered = [x for x in TARGET_OPTION_ORDER[target_col] if x in observed]
        extras = sorted([x for x in observed if x not in ordered])
        return ordered + extras

    return sorted(observed)


def make_letter_map(class_labels, row_seed):
    """Produce a reproducible per-row mapping of answer letters to class labels."""

    rng = np.random.default_rng(row_seed)
    letters = list(string.ascii_uppercase[:len(class_labels)])
    shuffled = list(class_labels)
    rng.shuffle(shuffled)

    return list(zip(letters, shuffled))


def build_profile_lines(row, target_col):
    """Build profile lines while hiding the target attribute."""

    lines = []

    for col, label in FEATURE_LABELS.items():
        if col == target_col:
            continue

        if col not in row.index:
            continue

        # PERWT is a survey weight, not a person attribute.
        # Raw AGE/INCTOT can leak the binned targets, so omit them.
        if col in ["AGE", "PERWT", "INCTOT"]:
            continue

        value = row[col]

        if pd.isna(value):
            continue

        lines.append(f"- {label}: {value}")

    return lines


def build_user_prompt(row, target_col, lettered_options):
    """Build the multiple-choice demographic prediction prompt."""

    profile_lines = build_profile_lines(row, target_col)
    target_label = FEATURE_LABELS.get(target_col, target_col)

    profile_text = "\n".join(profile_lines)

    answers_text = "\n".join([
        f"({letter}) {display_label(label)}"
        for letter, label in lettered_options
    ])

    return f"""Profile:

{profile_text}

Missing attribute: {target_label}

Answer options:
{answers_text}

Reply with only the letter of the correct option."""

## 6. Dataset loading utilities

In [ ]:
def dataset_path(method_or_split, target, kind=None):
    """Construct paths for train/eval parquet files."""

    if method_or_split in ["natural", "balanced", "weighted"]:
        return DATASET_ROOT / method_or_split / f"{target}.parquet"

    if method_or_split in ["validation", "test"]:
        if kind is None:
            raise ValueError("kind must be 'raw' or 'balanced' for validation/test datasets.")

        return DATASET_ROOT / method_or_split / kind / f"{target}.parquet"

    raise ValueError(f"Unknown dataset type: {method_or_split}")


def deterministic_row_seed(row_id, target_col, base_seed=42):
    """Create a stable per-row seed for option permutation."""

    text = f"{row_id}_{target_col}_{base_seed}"

    total = 0
    for i, ch in enumerate(text):
        total += (i + 1) * ord(ch)

    return int(total % (2**31 - 1))


def add_required_columns(df, target_col, base_seed=42):
    """Ensure row_id, true_{target}, and row_seed exist."""

    df = df.copy()

    if "row_id" not in df.columns:
        df = df.reset_index(drop=True)
        df["row_id"] = df.index

    true_col = f"true_{target_col}"

    if true_col not in df.columns:
        df[true_col] = df[target_col]

    if "row_seed" not in df.columns:
        df["row_seed"] = df["row_id"].apply(
            lambda x: deterministic_row_seed(x, target_col, base_seed)
        )

    return df


def maybe_subsample(df, max_rows, seed=42):
    """Subsample for smoke testing or faster validation."""

    if max_rows is None:
        return df.reset_index(drop=True)

    if len(df) <= max_rows:
        return df.reset_index(drop=True)

    return df.sample(n=max_rows, random_state=seed).reset_index(drop=True)


def load_target_dataset(method_or_split, target, kind=None, max_rows=None, seed=42):
    """Load one target dataset and add trainer-friendly columns."""

    path = dataset_path(method_or_split=method_or_split, target=target, kind=kind)

    if not path.exists():
        raise FileNotFoundError(f"Could not find dataset: {path}")

    df = pd.read_parquet(path)
    df = add_required_columns(df=df, target_col=target, base_seed=seed)
    df = maybe_subsample(df=df, max_rows=max_rows, seed=seed)

    return df


def load_train_datasets(method, targets, max_rows=None, seed=42):
    """Load training data for every target for one method."""

    return {
        target: load_target_dataset(
            method_or_split=method,
            target=target,
            kind=None,
            max_rows=max_rows,
            seed=seed,
        )
        for target in targets
    }


def load_eval_datasets(split, kind, targets, max_rows=None, seed=42):
    """Load validation or test data for every target."""

    return {
        target: load_target_dataset(
            method_or_split=split,
            target=target,
            kind=kind,
            max_rows=max_rows,
            seed=seed,
        )
        for target in targets
    }


def inspect_dataset_sizes(train_datasets, validation_datasets, test_datasets):
    """Print a quick size sanity check."""

    print("=" * 80)
    print("DATASET SIZE CHECK")
    print("=" * 80)

    for target in TARGETS:
        print(
            f"{target:<20} "
            f"train={len(train_datasets[target]):>6,} | "
            f"val={len(validation_datasets[target]):>6,} | "
            f"test={len(test_datasets[target]):>6,}"
        )

## 7. Load datasets for selected run

In [ ]:
run_config = RUN_CONFIGS[RUN_NAME]

METHOD = run_config["method"]
TARGET_ORDER = run_config["order"]

print("=" * 80)
print("SELECTED RUN")
print("=" * 80)
print("RUN_NAME:", RUN_NAME)
print("METHOD:", METHOD)
print("ORDER:", TARGET_ORDER)
print("DESCRIPTION:", run_config["description"])

train_datasets = load_train_datasets(
    method=METHOD,
    targets=TARGETS,
    max_rows=DEBUG_MAX_TRAIN_ROWS_PER_TARGET,
    seed=SEED,
)

validation_datasets = load_eval_datasets(
    split="validation",
    kind=VALIDATION_KIND,
    targets=TARGETS,
    max_rows=MAX_VALIDATION_ROWS_PER_TARGET,
    seed=SEED,
)

# inspect_dataset_sizes(
#     train_datasets=train_datasets,
#     validation_datasets=validation_datasets,
#     test_datasets=test_datasets,
# )

## 8. Tokenization and training dataset construction

In [ ]:
def build_messages(row, target_col, class_labels, include_answer=True):
    """Build chat messages for one row."""

    label_map = make_letter_map(
        class_labels=class_labels,
        row_seed=row["row_seed"],
    )

    true_answer = row[f"true_{target_col}"]

    answer_letter = None
    for letter, label in label_map:
        if label == true_answer:
            answer_letter = letter
            break

    if answer_letter is None:
        raise ValueError(
            f"True answer {true_answer} not found in label map for target {target_col}."
        )

    user_prompt = build_user_prompt(
        row=row,
        target_col=target_col,
        lettered_options=label_map,
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]

    if include_answer:
        messages.append({"role": "assistant", "content": answer_letter})

    return messages, label_map, answer_letter


def tokenize_one_training_example(row, target_col, class_labels, tokenizer, max_length=2048):
    """Convert a raw row into input_ids, attention_mask, labels, and sample_weight."""

    prompt_messages, _, _ = build_messages(
        row=row,
        target_col=target_col,
        class_labels=class_labels,
        include_answer=False,
    )

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    full_messages, _, _ = build_messages(
        row=row,
        target_col=target_col,
        class_labels=class_labels,
        include_answer=True,
    )

    full_text = tokenizer.apply_chat_template(
        full_messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    prompt_tokens = tokenizer(
    prompt_text,
    add_special_tokens=False,
    truncation=True,
    max_length=max_length,
)

    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=max_length,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    labels = list(input_ids)
    prompt_len = min(len(prompt_tokens["input_ids"]), len(labels))

    # Mask prompt tokens. Only answer/completion tokens affect loss.
    labels[:prompt_len] = [-100] * prompt_len

    sample_weight = float(row.get("class_weight", 1.0))

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
        "sample_weight": sample_weight,
    }


def build_lm_dataset(df, target_col, tokenizer, max_length=2048):
    """Build a Hugging Face Dataset for causal LM fine-tuning."""

    class_labels = get_target_classes(df, target_col)

    records = []

    for _, row in df.iterrows():
        records.append(
            tokenize_one_training_example(
                row=row,
                target_col=target_col,
                class_labels=class_labels,
                tokenizer=tokenizer,
                max_length=max_length,
            )
        )

    return Dataset.from_list(records)


class CausalLMDataCollator:

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, features):

        input_ids = [
            f["input_ids"]
            for f in features
        ]

        attention_mask = [
            f["attention_mask"]
            for f in features
        ]

        labels = [
            f["labels"]
            for f in features
        ]

        # ============================================
        # PAD INPUT IDS
        # ============================================

        batch = self.tokenizer.pad(
            {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
            },
            padding=True,
            return_tensors="pt",
        )

        # ============================================
        # PAD LABELS MANUALLY WITH -100
        # ============================================

        max_len = max(len(x) for x in labels)

        padded_labels = []

        for x in labels:

            padded = x + [-100] * (max_len - len(x))

            padded_labels.append(padded)

        batch["labels"] = torch.tensor(
            padded_labels,
            dtype=torch.long,
        )

        return batch

## 9. Weighted-loss trainer

In [ ]:
class WeightedCausalLMTrainer(Trainer):
    """Trainer that applies an example-level weight to token-level LM loss."""

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        sample_weight = inputs.pop("sample_weight", None)

        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # Shift for causal LM loss.
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = torch.nn.CrossEntropyLoss(
            reduction="none",
            ignore_index=-100,
        )

        token_losses = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        ).view(shift_labels.size())

        token_mask = (shift_labels != -100).float()

        per_example_loss = (
            token_losses * token_mask
        ).sum(dim=1) / token_mask.sum(dim=1).clamp(min=1.0)

        if sample_weight is not None:
            sample_weight = sample_weight.to(per_example_loss.device)
            loss = (per_example_loss * sample_weight).mean()
        else:
            loss = per_example_loss.mean()

        return (loss, outputs) if return_outputs else loss

## 10. Model loading and checkpoint utilities

In [ ]:
# ============================================================
# LOAD BASE MODEL + LORA
# ============================================================

def load_base_lora_model(model_name, use_4bit=True):
    """
    Load the base model and attach fresh LoRA adapters.
    """

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ========================================================
    # 4-BIT QUANTIZED MODEL
    # ========================================================

    if use_4bit:

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

        print("bnb_config:", bnb_config)
        print("use_4bit:", use_4bit)

        base_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
          device_map={"": 0},
            max_memory={0: "36GiB"},
            trust_remote_code=True,
        )
        print(next(base_model.parameters()).dtype)



        # ----------------------------------------------------
        # IMPORTANT FOR MEMORY SAVINGS
        # ----------------------------------------------------

        base_model.gradient_checkpointing_enable()

        base_model = prepare_model_for_kbit_training(
            base_model
        )

    # ========================================================
    # FULL PRECISION MODEL
    # ========================================================

    else:

        dtype = (
            torch.bfloat16
            if torch.cuda.is_available()
            else torch.float32
        )

        base_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            device_map={"": 0}
            if torch.cuda.is_available()
            else None,
             max_memory={0: "36GiB"},
            trust_remote_code=True,
        )

        base_model.gradient_checkpointing_enable()

    # ========================================================
    # LORA CONFIG
    # ========================================================

    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
        ],
    )

    model = get_peft_model(
        base_model,
        peft_config,
    )

    model.print_trainable_parameters()

    return model, tokenizer


# ============================================================
# LOAD MODEL FROM SAVED ADAPTER
# ============================================================

def load_model_from_adapter(
    model_name,
    adapter_path,
    use_4bit=True,
):
    """
    Resume a sequential run from a saved LoRA adapter.
    """

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ========================================================
    # 4-BIT MODEL
    # ========================================================

    if use_4bit:

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )
        print("bnb_config:", bnb_config)
        print("use_4bit:", use_4bit)

        base_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map={"":0},
            max_memory={0: "36GiB"},
            trust_remote_code=True,
        )

        base_model.gradient_checkpointing_enable()

        base_model = prepare_model_for_kbit_training(
            base_model
        )

    # ========================================================
    # FULL PRECISION MODEL
    # ========================================================

    else:

        dtype = (
            torch.bfloat16
            if torch.cuda.is_available()
            else torch.float32
        )

        base_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=dtype,
            device_map={"": 0}
            if torch.cuda.is_available()
            else None,
            max_memory={0: "36GiB"},
            trust_remote_code=True,
        )

        base_model.gradient_checkpointing_enable()

    # ========================================================
    # LOAD LORA ADAPTER
    # ========================================================

    model = PeftModel.from_pretrained(
        base_model,
        adapter_path,
        is_trainable=True,
    )

    model.print_trainable_parameters()

    return model, tokenizer


# ============================================================
# SAVE CHECKPOINT
# ============================================================

def save_lora_checkpoint(
    model,
    tokenizer,
    save_path,
):
    """
    Save LoRA adapter + tokenizer.
    """

    save_path = Path(save_path)

    save_path.mkdir(
        parents=True,
        exist_ok=True,
    )

    model.save_pretrained(save_path)

    tokenizer.save_pretrained(save_path)

    print(f"Saved LoRA checkpoint → {save_path}")


# ============================================================
# CLEANUP GPU MEMORY
# ============================================================

def cleanup_memory():
    """
    Free GPU memory between heavy operations.
    """

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        torch.cuda.ipc_collect()

## 11. Evaluation with answer-letter log probabilities

In [ ]:
def get_answer_token_id(tokenizer, letter):
    """Get the first token id for an answer letter."""

    token_ids = tokenizer.encode(letter, add_special_tokens=False)

    if len(token_ids) == 0:
        raise ValueError(f"No token id found for letter {letter}")

    return token_ids[0]


def evaluate_with_logprobs(model, tokenizer, eval_df, target_col, max_rows=None, seed=42):
    """Evaluate a target by extracting logprobs over answer-letter options."""

    model.eval()

    eval_df = maybe_subsample(df=eval_df, max_rows=max_rows, seed=seed)
    class_labels = get_target_classes(eval_df, target_col)

    rows = []

    for _, row in eval_df.iterrows():
        messages, label_map, _ = build_messages(
            row=row,
            target_col=target_col,
            class_labels=class_labels,
            include_answer=False,
        )

        letter_to_class = {letter: label for letter, label in label_map}
        answer_letters = list(letter_to_class.keys())

        answer_token_ids = {
            letter: get_answer_token_id(tokenizer, letter)
            for letter in answer_letters
        }

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits[:, -1, :]
        log_probs = torch.log_softmax(logits, dim=-1)

        result = row.to_dict()
        class_probs = {}

        for letter, token_id in answer_token_ids.items():
            class_name = letter_to_class[letter]

            lp = log_probs[0, token_id].item()
            prob = float(np.exp(lp))

            result[f"logprob_{class_name}"] = lp
            result[f"prob_{class_name}"] = prob

            class_probs[class_name] = prob

        pred_class = max(class_probs, key=class_probs.get)
        true_col = f"true_{target_col}"

        result[f"pred_{target_col}"] = pred_class
        result["correct"] = pred_class == row[true_col]

        rows.append(result)

    return pd.DataFrame(rows)


def summarize_eval_results(results_df, target_col):
    """Create compact overall and per-class metrics."""

    true_col = f"true_{target_col}"

    overall = {
        "target": target_col,
        "n": len(results_df),
        "accuracy": results_df["correct"].mean(),
    }

    per_class = (
        results_df
        .groupby(true_col, observed=True)
        .agg(n=("correct", "count"), accuracy=("correct", "mean"))
        .reset_index()
        .rename(columns={true_col: "class"})
    )

    per_class["target"] = target_col

    return overall, per_class

## 12. Training/evaluation pipeline

In [ ]:
def train_one_target(model, tokenizer, train_df, target_col, run_dir, step_idx):
    """Train the current model on one target dataset."""

    print("\n")
    print("=" * 80)
    print(f"TRAINING STEP {step_idx}: {target_col}")
    print("=" * 80)

    train_dataset = build_lm_dataset(
        df=train_df,
        target_col=target_col,
        tokenizer=tokenizer,
    )

    output_dir = run_dir / "trainer_tmp" / f"step_{step_idx:02d}_{target_col}"

    training_args = TrainingArguments(
        output_dir=str(output_dir),
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        max_steps=MAX_STEPS_PER_TARGET,
        logging_steps=1 if SMOKE_TEST else 25,
        save_strategy="no",
        report_to="none",
        fp16=False,
        bf16=True,
        remove_unused_columns=False,
    )

    trainer = WeightedCausalLMTrainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        data_collator=CausalLMDataCollator(tokenizer),
    )

    trainer.train()

    return trainer.model


def evaluate_all_targets(
    model,
    tokenizer,
    eval_datasets,
    run_dir,
    split_name,
    checkpoint_label,
    train_step_idx,
    trained_target,
    max_rows_per_target=None,
):

    import gc
    import json
    import torch
    import pandas as pd
    import matplotlib.pyplot as plt

    # ========================================================
    # ONLY EVALUATE TARGETS SEEN SO FAR
    # ========================================================

    # eval_targets = TARGET_ORDER[:train_step_idx]

    eval_targets = [trained_target]

    if train_step_idx == len(TARGET_ORDER):
      eval_targets = TARGET_ORDER

    # ========================================================
    # SAVE DIRECTORY
    # ========================================================

    results_root = (
        run_dir
        / "evaluations"
        / split_name
        / checkpoint_label
    )

    results_root.mkdir(
        parents=True,
        exist_ok=True,
    )

    # ========================================================
    # STORAGE
    # ========================================================

    overall_rows = []
    per_class_rows = []

    # ========================================================
    # HEADER
    # ========================================================

    print("\n")
    print("=" * 80)
    print(
        f"STEP {train_step_idx} | "
        f"TRAINED ON: {trained_target}"
    )
    print("=" * 80)

    # ========================================================
    # EVALUATE EACH TARGET
    # ========================================================

    for eval_target in eval_targets:

        print(f"Evaluating → {eval_target}")

        # ----------------------------------------------------
        # RUN EVALUATION
        # ----------------------------------------------------

        results_df = evaluate_with_logprobs(
            model=model,
            tokenizer=tokenizer,
            eval_df=eval_datasets[eval_target],
            target_col=eval_target,
            max_rows=max_rows_per_target,
            seed=SEED,
        )

        # ----------------------------------------------------
        # ADD METADATA
        # ----------------------------------------------------

        results_df["run_name"] = RUN_NAME
        results_df["method"] = METHOD
        results_df["checkpoint_label"] = checkpoint_label
        results_df["train_step_idx"] = train_step_idx
        results_df["trained_target"] = trained_target
        results_df["eval_target"] = eval_target
        results_df["target_order"] = json.dumps(TARGET_ORDER)

        # ----------------------------------------------------
        # SAVE FULL PREDICTIONS
        # ----------------------------------------------------

        save_path = (
            results_root
            / f"eval_{eval_target}.parquet"
        )

        results_df.to_parquet(save_path)

        # ----------------------------------------------------
        # SUMMARY METRICS
        # ----------------------------------------------------

        overall, per_class = summarize_eval_results(
            results_df=results_df,
            target_col=eval_target,
        )

        # ----------------------------------------------------
        # OVERALL METADATA
        # ----------------------------------------------------

        overall.update({
            "run_name": RUN_NAME,
            "method": METHOD,
            "split": split_name,
            "checkpoint_label": checkpoint_label,
            "train_step_idx": train_step_idx,
            "trained_target": trained_target,
        })

        # ----------------------------------------------------
        # PER-CLASS METADATA
        # ----------------------------------------------------

        per_class["run_name"] = RUN_NAME
        per_class["method"] = METHOD
        per_class["split"] = split_name
        per_class["checkpoint_label"] = checkpoint_label
        per_class["train_step_idx"] = train_step_idx
        per_class["trained_target"] = trained_target

        # ----------------------------------------------------
        # STORE
        # ----------------------------------------------------

        overall_rows.append(overall)
        per_class_rows.append(per_class)

    # ========================================================
    # BUILD TABLES
    # ========================================================

    overall_df = pd.DataFrame(overall_rows)

    per_class_df = pd.concat(
        per_class_rows,
        ignore_index=True,
    )

    # ========================================================
    # SAVE SUMMARY TABLES
    # ========================================================

    overall_df.to_parquet(
        results_root / "overall_metrics.parquet"
    )

    overall_df.to_csv(
        results_root / "overall_metrics.csv",
        index=False,
    )

    per_class_df.to_parquet(
        results_root / "per_class_metrics.parquet"
    )

    per_class_df.to_csv(
        results_root / "per_class_metrics.csv",
        index=False,
    )

    # ========================================================
    # QUICK CONSOLE SUMMARY
    # ========================================================

    summary_cols = [
        "target",
        "accuracy",
    ]

    if "f1" in overall_df.columns:
        summary_cols.append("f1")

    if "kappa" in overall_df.columns:
        summary_cols.append("kappa")

    print("\n")
    print("CURRENT ACCURACIES")
    print("-" * 80)

    display(
        overall_df[summary_cols]
        .sort_values("target")
        .round(3)
    )

    # ========================================================
    # CURRENT TARGET SUBGROUP PLOT
    # ========================================================

    current_target_df = (
        per_class_df[
            per_class_df["target"]
            == trained_target
        ]
        .sort_values("accuracy")
    )

    if len(current_target_df) > 0:

        plt.figure(figsize=(10, 5))

        plt.bar(
            current_target_df["class"].astype(str),
            current_target_df["accuracy"],
        )

        plt.ylim(0, 1)

        plt.xlabel("Class")
        plt.ylabel("Accuracy")

        plt.title(
            f"{trained_target} subgroup accuracy"
        )

        plt.xticks(rotation=45)

        plt.show()

    # ========================================================
    # MEMORY CLEANUP
    # ========================================================

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return overall_df, per_class_df

def write_run_metadata(run_dir):
    """Save experiment metadata for reproducibility."""

    metadata = {
        "run_name": RUN_NAME,
        "method": METHOD,
        "target_order": TARGET_ORDER,
        "validation_kind": VALIDATION_KIND,
        "test_kind": TEST_KIND,
        "model_name": MODEL_NAME,
        "smoke_test": SMOKE_TEST,
        "debug_max_train_rows_per_target": DEBUG_MAX_TRAIN_ROWS_PER_TARGET,
        "max_validation_rows_per_target": MAX_VALIDATION_ROWS_PER_TARGET,
        "max_test_rows_per_target": MAX_TEST_ROWS_PER_TARGET,
        "num_epochs": NUM_EPOCHS,
        "max_steps_per_target": MAX_STEPS_PER_TARGET,
        "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "learning_rate": LEARNING_RATE,
        "use_4bit": USE_4BIT,
        "seed": SEED,
        "created_at": datetime.now().isoformat(),
    }

    with open(run_dir / "run_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    pd.DataFrame([
        {"step_idx": i + 1, "target": target}
        for i, target in enumerate(TARGET_ORDER)
    ]).to_csv(run_dir / "training_order.csv", index=False)


def run_sequential_experiment(resume_from_adapter_path=None, start_step_idx=1):
    """Run one sequential experiment."""

    set_seed(SEED)

    run_dir = EXPERIMENT_ROOT / RUN_NAME
    run_dir.mkdir(parents=True, exist_ok=True)

    for subdir in ["checkpoints", "evaluations", "logs", "trainer_tmp"]:
        (run_dir / subdir).mkdir(parents=True, exist_ok=True)

    write_run_metadata(run_dir)

    print("=" * 80)
    print("RUN DIRECTORY")
    print("=" * 80)
    print(run_dir)

    if resume_from_adapter_path is None:
        model, tokenizer = load_base_lora_model(
            model_name=MODEL_NAME,
            use_4bit=USE_4BIT,
        )
    else:
        model, tokenizer = load_model_from_adapter(
            model_name=MODEL_NAME,
            adapter_path=resume_from_adapter_path,
            use_4bit=USE_4BIT,
        )


    for step_idx, train_target in enumerate(TARGET_ORDER, start=1):
        if step_idx < start_step_idx:
            print(f"Skipping step {step_idx}: {train_target}")
            continue

        model = train_one_target(
            model=model,
            tokenizer=tokenizer,
            train_df=train_datasets[train_target],
            target_col=train_target,
            run_dir=run_dir,
            step_idx=step_idx,
        )

        checkpoint_label = f"step_{step_idx:02d}_after_{train_target}"

        if SAVE_INTERMEDIATE_CHECKPOINTS:
            checkpoint_path = run_dir / "checkpoints" / checkpoint_label
            save_lora_checkpoint(model, tokenizer, checkpoint_path)

        evaluate_all_targets(
            model=model,
            tokenizer=tokenizer,
            eval_datasets=validation_datasets,
            run_dir=run_dir,
            split_name=f"validation_{VALIDATION_KIND}",
            checkpoint_label=checkpoint_label,
            train_step_idx=step_idx,
            trained_target=train_target,
            max_rows_per_target=MAX_VALIDATION_ROWS_PER_TARGET,
        )

        cleanup_memory()


    if SAVE_FINAL_MERGED_MODEL:
        final_path = run_dir / "final_merged_model"
        merged = model.merge_and_unload()
        merged.save_pretrained(final_path)
        tokenizer.save_pretrained(final_path)
        print(f"Saved merged model → {final_path}")

    print("\nDONE")
    print(run_dir)

    return run_dir

## 13. Run smoke test

Run this first with `SMOKE_TEST = True`. It should finish quickly and prove the pipeline works.

In [ ]:
# run_dir = run_sequential_experiment(
#     resume_from_adapter_path=None,
#     start_step_idx=1,
# )

## 14. Full run example

After smoke testing:

1. Set `SMOKE_TEST = False`
2. Set `RUN_NAME` to the run you want
3. Restart runtime if needed
4. Re-run setup/data-loading cells
5. Run the function below

In [ ]:
# ============================================================
# CONFIGURE RUN
# ============================================================

def configure_run(
    run_name,
    smoke_test=False,
):
    """
    Configure experiment settings for a selected run.

    This updates:
    - RUN_NAME
    - METHOD
    - TARGET_ORDER
    - model settings
    """

    global RUN_NAME
    global METHOD
    global TARGET_ORDER
    global MODEL_NAME
    global SMOKE_TEST
    global USE_4BIT

    global DEBUG_MAX_TRAIN_ROWS_PER_TARGET
    global DEBUG_MAX_EVAL_ROWS_PER_TARGET

    global NUM_EPOCHS
    global MAX_STEPS_PER_TARGET

    global PER_DEVICE_BATCH_SIZE
    global GRAD_ACCUM
    global LEARNING_RATE

    global MAX_VALIDATION_ROWS_PER_TARGET
    global MAX_TEST_ROWS_PER_TARGET

    # --------------------------------------------------------
    # RUN CONFIG
    # --------------------------------------------------------

    RUN_NAME = run_name

    run_config = RUN_CONFIGS[RUN_NAME]

    METHOD = run_config["method"]
    TARGET_ORDER = run_config["order"]

    # --------------------------------------------------------
    # SMOKE TEST SETTINGS
    # --------------------------------------------------------

    SMOKE_TEST = smoke_test

    if SMOKE_TEST:

        MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

        DEBUG_MAX_TRAIN_ROWS_PER_TARGET = 24
        DEBUG_MAX_EVAL_ROWS_PER_TARGET = 24

        NUM_EPOCHS = 1
        MAX_STEPS_PER_TARGET = 2

        PER_DEVICE_BATCH_SIZE = 1
        GRAD_ACCUM = 1
        LEARNING_RATE = 2e-4

        USE_4BIT = False

    else:

        MODEL_NAME = "Qwen/Qwen2.5-14B-Instruct"


        DEBUG_MAX_TRAIN_ROWS_PER_TARGET = None
        DEBUG_MAX_EVAL_ROWS_PER_TARGET = None

        NUM_EPOCHS = 1
        MAX_STEPS_PER_TARGET = -1

        PER_DEVICE_BATCH_SIZE = 16
        GRAD_ACCUM = 2
        LEARNING_RATE = 2e-4

        USE_4BIT = True

    # --------------------------------------------------------
    # EVAL SETTINGS
    # --------------------------------------------------------

    MAX_VALIDATION_ROWS_PER_TARGET = (
        DEBUG_MAX_EVAL_ROWS_PER_TARGET
    )

    MAX_TEST_ROWS_PER_TARGET = (
        DEBUG_MAX_EVAL_ROWS_PER_TARGET
    )

    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print("\n")
    print("=" * 80)
    print("RUN CONFIGURED")
    print("=" * 80)

    print(f"RUN_NAME:   {RUN_NAME}")
    print(f"METHOD:     {METHOD}")
    print(f"ORDER:      {TARGET_ORDER}")
    print(f"SMOKE_TEST: {SMOKE_TEST}")
    print(f"MODEL:      {MODEL_NAME}")

In [ ]:
# ============================================================
# LOAD RUN DATASETS
# ============================================================

def load_run_datasets():
    """
    Reload datasets for the currently selected run.
    """

    global train_datasets
    global validation_datasets
    global test_datasets

    train_datasets = load_train_datasets(
        method=METHOD,
        targets=CANONICAL_ORDER,
        max_rows=DEBUG_MAX_TRAIN_ROWS_PER_TARGET,
        seed=SEED,
    )

    validation_datasets = load_eval_datasets(
        split="validation",
        kind=VALIDATION_KIND,
        targets=CANONICAL_ORDER,
        max_rows=MAX_VALIDATION_ROWS_PER_TARGET,
        seed=SEED,
    )

    test_datasets = load_eval_datasets(
        split="test",
        kind=TEST_KIND,
        targets=CANONICAL_ORDER,
        max_rows=MAX_TEST_ROWS_PER_TARGET,
        seed=SEED,
    )

    inspect_dataset_sizes(
        train_datasets=train_datasets,
        validation_datasets=validation_datasets,
        test_datasets=test_datasets,
    )

    print("\nDatasets loaded.")

In [ ]:
# configure_run(
#     run_name="natural_canonical",
#     smoke_test=False,
# )

# load_run_datasets()

In [ ]:
# run_dir = run_sequential_experiment(
#     resume_from_adapter_path=None,
#     start_step_idx=1,

# )

In [ ]:
use_4bit = True

In [ ]:
# configure_run(
#     run_name="balanced_canonical",
#     smoke_test=False,
# )

# load_run_datasets()
# SAVE_FINAL_MERGED_MODEL = True

In [ ]:
# run_dir = run_sequential_experiment(
#     resume_from_adapter_path=None,
#     start_step_idx=1,

# )

In [ ]:
configure_run(
    run_name="weighted_canonical",
    smoke_test=False,
)

load_run_datasets()

In [ ]:
run_dir = run_sequential_experiment(
    resume_from_adapter_path=None,
    start_step_idx=1,

)

In [ ]:
# # ============================================================
# # AUTO TERMINATE COLAB RUNTIME
# # ============================================================

# import os
# import time

# print("\n")
# print("=" * 80)
# print("EXPERIMENT FINISHED")
# print("=" * 80)

# print("Shutting down runtime in 60 seconds...")

# time.sleep(60)

# os.kill(os.getpid(), 9)

## 17. Collect metrics for one run

In [ ]:
# def collect_metrics_for_run(run_name):
#     """Collect all overall/per-class metrics saved for a run."""

#     run_dir = EXPERIMENT_ROOT / run_name / "evaluations"

#     overall_files = list(run_dir.rglob("overall_metrics.csv"))
#     per_class_files = list(run_dir.rglob("per_class_metrics.csv"))

#     overall_parts = []
#     for path in overall_files:
#         df_part = pd.read_csv(path)
#         df_part["source_file"] = str(path)
#         overall_parts.append(df_part)

#     per_class_parts = []
#     for path in per_class_files:
#         df_part = pd.read_csv(path)
#         df_part["source_file"] = str(path)
#         per_class_parts.append(df_part)

#     overall_df = pd.concat(overall_parts, ignore_index=True) if overall_parts else pd.DataFrame()
#     per_class_df = pd.concat(per_class_parts, ignore_index=True) if per_class_parts else pd.DataFrame()

#     return overall_df, per_class_df


# overall_metrics, per_class_metrics = collect_metrics_for_run(RUN_NAME)

# display(overall_metrics.head())
# display(per_class_metrics.head())

## 18. Forgetting / recency summary

In [ ]:
def compute_forgetting(overall_metrics):
    """Compute forgetting = peak validation accuracy - final validation accuracy."""

    if overall_metrics.empty:
        return pd.DataFrame()

    val_df_metrics = overall_metrics[
        overall_metrics["split"].str.startswith("validation")
    ].copy()

    trained_df = val_df_metrics[val_df_metrics["train_step_idx"] > 0].copy()

    if trained_df.empty:
        return pd.DataFrame()

    peak = (
        trained_df
        .groupby(["run_name", "method", "target"], observed=True)["accuracy"]
        .max()
        .reset_index()
        .rename(columns={"accuracy": "peak_accuracy"})
    )

    max_step = trained_df["train_step_idx"].max()

    final = (
        trained_df[trained_df["train_step_idx"] == max_step]
        [["run_name", "method", "target", "accuracy"]]
        .rename(columns={"accuracy": "final_accuracy"})
    )

    out = peak.merge(final, on=["run_name", "method", "target"], how="left")
    out["forgetting"] = out["peak_accuracy"] - out["final_accuracy"]

    return out.sort_values("forgetting", ascending=False)


forgetting_df = compute_forgetting(overall_metrics)
display(forgetting_df)

## 19. Combine all completed runs

In [ ]:
def collect_all_runs(run_names):
    """Collect metrics across multiple completed runs."""

    overall_all = []
    per_class_all = []

    for rn in run_names:
        overall_df, per_class_df = collect_metrics_for_run(rn)

        if not overall_df.empty:
            overall_all.append(overall_df)

        if not per_class_df.empty:
            per_class_all.append(per_class_df)

    overall_all = pd.concat(overall_all, ignore_index=True) if overall_all else pd.DataFrame()
    per_class_all = pd.concat(per_class_all, ignore_index=True) if per_class_all else pd.DataFrame()

    return overall_all, per_class_all


# all_run_names = [
#     "natural_canonical",
#     "balanced_canonical",
#     "weighted_canonical",
#     "natural_reverse",
#     "natural_random1",
# ]
#
# all_overall, all_per_class = collect_all_runs(all_run_names)
# display(all_overall)
# display(compute_forgetting(all_overall))